# Yahoo Price Drift + COT Positioning Analysis

This notebook loads `data/centralData/yahoo_cot_full_outer_by_date.csv`, detects where Arabica Coffee C Yahoo price behavior drifts, and overlays the relevant COT positioning lines on the same timeline.

The purpose is explanatory rather than predictive: find the timestamps where price behavior changed sharply, then inspect whether COT positioning, open interest, speculative length, commercial hedging, swap positioning, or non-commercial positioning were changing around those same periods.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_PATH = ROOT / "data" / "centralData" / "yahoo_cot_full_outer_by_date.csv"
FFILL_DATA_PATH = ROOT / "data" / "centralData" / "yahoo_cot_full_outer_by_date_cot_ffill.csv"
OUTPUT_DIR = ROOT / "project" / "artifacts" / "outputs"
PLOT_DIR = ROOT / "project" / "artifacts" / "plots"
OUTPUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("Data path:", DATA_PATH)
print("Forward-filled COT path:", FFILL_DATA_PATH)

## 2. Load Yahoo + COT central data

In [ ]:
raw = pd.read_csv(DATA_PATH, low_memory=False)
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
raw["cot_report_date"] = pd.to_datetime(raw["cot_report_date"], errors="coerce")
raw = raw.sort_values("Date").reset_index(drop=True)

print("Rows, columns:", raw.shape)
print("Date range:", raw["Date"].min(), "to", raw["Date"].max())
print("Yahoo price rows:", raw["Close"].notna().sum())
print("COT rows:", raw["cot_report_date"].notna().sum())
display(raw[["Date", "date_match_status", "cot_report_date", "Open", "High", "Low", "Close", "Volume"]].head())

## 2A. Build and load the forward-filled COT daily dataset

The original outer-joined file has weekly COT rows and many daily Yahoo rows with blank COT columns. For COT-vs-price plotting, we use the forward-filled copy created by `project/scripts/fill_cot_forward_daily.py`.

This fills only COT/reporting columns. Yahoo price, return, and future-target columns are left unchanged. The audit columns `cot_fill_status` and `cot_days_since_report` show whether a row is original COT data or carried forward from the latest report.

In [ ]:
if not FFILL_DATA_PATH.exists():
    subprocess.run([
        sys.executable,
        "project/scripts/fill_cot_forward_daily.py",
        "--input", str(DATA_PATH),
        "--output", str(FFILL_DATA_PATH),
    ], check=True)

filled = pd.read_csv(FFILL_DATA_PATH, low_memory=False)
filled["Date"] = pd.to_datetime(filled["Date"], errors="coerce")
filled["cot_report_date"] = pd.to_datetime(filled["cot_report_date"], errors="coerce")
if "cot_release_date" in filled.columns:
    filled["cot_release_date"] = pd.to_datetime(filled["cot_release_date"], errors="coerce")
filled = filled.sort_values("Date").reset_index(drop=True)

price_cols = ["Close", "High", "Low", "Open", "Volume", "return_1d", "return_5d", "future_return_5d"]
print("Filled rows, columns:", filled.shape)
print("COT fill status counts:")
display(filled["cot_fill_status"].value_counts(dropna=False).to_frame("rows"))
print("Yahoo price/return/target columns unchanged:", raw[price_cols].equals(filled[price_cols]))
display(filled[["Date", "date_match_status", "cot_report_date", "cot_fill_status", "cot_days_since_report", "Close", "Open_Interest_All"]].head(12))

## 3. Build price drift metrics

Drift here means a significant change in the Yahoo price process. This notebook flags dates where at least one of these conditions occurs:

- 5-trading-day return has a large 252-day z-score
- 20-trading-day return has a large 252-day z-score
- 20-day realized volatility has a large 252-day z-score
- price is far away from the 63-day moving average

These rules are intentionally transparent so the drift flags are easy to audit.

In [ ]:
price = raw.loc[raw["Close"].notna(), ["Date", "Open", "High", "Low", "Close", "Volume"]].copy()
price = price.sort_values("Date").drop_duplicates("Date", keep="last").reset_index(drop=True)

price["return_1d"] = price["Close"].pct_change(1)
price["return_5d"] = price["Close"].pct_change(5)
price["return_20d"] = price["Close"].pct_change(20)
price["realized_vol_20d"] = price["return_1d"].rolling(20, min_periods=10).std() * np.sqrt(252)
price["ma_63"] = price["Close"].rolling(63, min_periods=30).mean()
price["close_vs_ma_63"] = price["Close"] / price["ma_63"] - 1

def rolling_zscore(series, window=252, min_periods=80):
    rolling_mean = series.rolling(window, min_periods=min_periods).mean()
    rolling_std = series.rolling(window, min_periods=min_periods).std()
    return (series - rolling_mean) / rolling_std.replace(0, np.nan)

price["return_5d_z"] = rolling_zscore(price["return_5d"])
price["return_20d_z"] = rolling_zscore(price["return_20d"])
price["realized_vol_20d_z"] = rolling_zscore(price["realized_vol_20d"])
price["trend_distance_z"] = rolling_zscore(price["close_vs_ma_63"])

threshold = 2.0
price["price_drift_score"] = price[[
    "return_5d_z", "return_20d_z", "realized_vol_20d_z", "trend_distance_z"
]].abs().max(axis=1)
price["price_drift_flag"] = price["price_drift_score"] >= threshold

display(price.tail())
print("Drift-flagged trading days:", int(price["price_drift_flag"].sum()))

## 4. Collapse drift days into drift events

A single market episode can trigger many consecutive drift days. This step groups nearby drift-flagged days into events and keeps the strongest date in each event.

In [ ]:
def make_drift_events(price_df, max_gap_days=10):
    flagged = price_df.loc[price_df["price_drift_flag"]].copy()
    if flagged.empty:
        return pd.DataFrame()

    event_ids = []
    event_id = 0
    previous_date = None
    for date in flagged["Date"]:
        if previous_date is None or (date - previous_date).days > max_gap_days:
            event_id += 1
        event_ids.append(event_id)
        previous_date = date
    flagged["event_id"] = event_ids

    rows = []
    for event_id, group in flagged.groupby("event_id"):
        peak = group.loc[group["price_drift_score"].idxmax()]
        rows.append({
            "event_id": event_id,
            "event_start": group["Date"].min(),
            "event_end": group["Date"].max(),
            "peak_date": peak["Date"],
            "peak_close": peak["Close"],
            "peak_drift_score": peak["price_drift_score"],
            "return_5d": peak["return_5d"],
            "return_20d": peak["return_20d"],
            "return_5d_z": peak["return_5d_z"],
            "return_20d_z": peak["return_20d_z"],
            "realized_vol_20d_z": peak["realized_vol_20d_z"],
            "trend_distance_z": peak["trend_distance_z"],
            "trading_days_in_event": len(group),
        })
    return pd.DataFrame(rows).sort_values("peak_drift_score", ascending=False)

drift_events = make_drift_events(price)
drift_events.to_csv(OUTPUT_DIR / "yahoo_price_drift_events.csv", index=False)
display(drift_events.head(30))

## 5. Append filled daily COT data for plotting

The filled dataset carries each COT report forward into the daily Yahoo rows between reports. This gives the plots a daily COT line instead of sparse weekly points.

The notebook still keeps the timing explicit: `cot_report_date` is the report date carried forward, `cot_days_since_report` tells how old the carried report is, and `cot_fill_status` shows whether the row was an original COT row or forward-filled.

In [ ]:
cot_features = [
    "Open_Interest_All",
    "managed_money_net",
    "managed_money_net_pct_oi",
    "producer_merchant_net",
    "commercial_net",
    "noncommercial_net",
    "nonreportable_net",
    "swap_dealer_net",
    "other_reportable_net",
    "managed_money_weekly_net_change",
    "commercial_weekly_net_change",
    "noncommercial_weekly_net_change",
    "Pct_of_OI_M_Money_Long_All",
    "Pct_of_OI_M_Money_Short_All",
    "Pct_of_OI_Comm_Long_All",
    "Pct_of_OI_Comm_Short_All",
    "Pct_of_OI_NonComm_Long_All",
    "Pct_of_OI_NonComm_Short_All",
]
cot_features = [c for c in cot_features if c in filled.columns]

daily_cot = filled[[
    "Date", "cot_report_date", "cot_fill_status", "cot_days_since_report", *cot_features
]].copy()
daily_cot = daily_cot.sort_values("Date").drop_duplicates("Date", keep="last")
daily_cot["cot_release_date_estimate"] = daily_cot["cot_report_date"] + pd.Timedelta(days=3)

aligned = price.merge(daily_cot, on="Date", how="left")

print("Aligned frame using filled daily COT:", aligned.shape)
print("COT fill status in aligned price rows:")
display(aligned["cot_fill_status"].value_counts(dropna=False).to_frame("rows"))
display(aligned[["Date", "Close", "cot_report_date", "cot_release_date_estimate", "cot_fill_status", "cot_days_since_report", *cot_features[:6]]].tail())

## 6. Correlate COT with price drift and returns

In [ ]:
corr_rows = []
for feature in cot_features:
    x = pd.to_numeric(aligned[feature], errors="coerce")
    corr_rows.append({
        "cot_feature": feature,
        "corr_with_return_5d": x.corr(aligned["return_5d"]),
        "corr_with_return_20d": x.corr(aligned["return_20d"]),
        "corr_with_abs_return_5d_z": x.corr(aligned["return_5d_z"].abs()),
        "corr_with_price_drift_score": x.corr(aligned["price_drift_score"]),
        "missing_rate": x.isna().mean(),
    })

cot_corr = pd.DataFrame(corr_rows)
cot_corr["max_abs_corr"] = cot_corr[[
    "corr_with_return_5d", "corr_with_return_20d", "corr_with_abs_return_5d_z", "corr_with_price_drift_score"
]].abs().max(axis=1)
cot_corr = cot_corr.sort_values("max_abs_corr", ascending=False)
cot_corr.to_csv(OUTPUT_DIR / "cot_price_drift_correlations.csv", index=False)
display(cot_corr)

In [ ]:
plt.figure(figsize=(10, 7))
top_corr = cot_corr.head(15).sort_values("max_abs_corr")
plt.barh(top_corr["cot_feature"], top_corr["max_abs_corr"])
plt.title("Top COT Correlations With Yahoo Price Drift Metrics")
plt.xlabel("Max absolute correlation across drift/return metrics")
plt.tight_layout()
plt.savefig(PLOT_DIR / "cot_price_drift_correlations.png", dpi=180)
plt.show()

## 7. Same-plot timeline: Yahoo price drift + COT lines

To put price and COT on the same chart, the lines below are standardized z-scores. Vertical lines mark the strongest price drift events. This makes it easier to see whether positioning was stretched, reversing, or accelerating near price drift timestamps.

In [ ]:
def zscore_for_plot(series):
    series = pd.to_numeric(series, errors="coerce")
    return (series - series.mean(skipna=True)) / series.std(skipna=True)

plot_features = [
    "Close",
    "Open_Interest_All",
    "managed_money_net_pct_oi",
    "commercial_net",
    "noncommercial_net",
    "swap_dealer_net",
]
plot_features = [c for c in plot_features if c in aligned.columns]
plot_df = aligned[["Date", *plot_features, "price_drift_score", "price_drift_flag"]].copy()
for col in plot_features:
    plot_df[f"{col}_z"] = zscore_for_plot(plot_df[col])

top_events = drift_events.head(20).copy()

plt.figure(figsize=(18, 8))
for col in plot_features:
    linewidth = 2.4 if col == "Close" else 1.4
    alpha = 0.95 if col == "Close" else 0.75
    plt.plot(plot_df["Date"], plot_df[f"{col}_z"], label=f"{col} z-score", linewidth=linewidth, alpha=alpha)

for _, event in top_events.iterrows():
    color = "crimson" if event["return_20d"] < 0 else "seagreen"
    plt.axvline(event["peak_date"], color=color, alpha=0.25, linewidth=1)

plt.axhline(0, color="black", linewidth=0.8)
plt.title("Yahoo Coffee C Price Drift With COT Positioning Lines")
plt.ylabel("Standardized value")
plt.xlabel("Timestamp")
plt.legend(loc="upper left", ncol=2)
plt.tight_layout()
plt.savefig(PLOT_DIR / "yahoo_price_drift_with_cot_lines.png", dpi=180)
plt.show()

## 8. Zoom into the strongest drift events

The full timeline can get crowded. This section creates one plot per strongest drift event, using a window around the peak timestamp.

In [ ]:
def plot_event_window(event_row, window_days=180, features=None):
    features = features or plot_features
    start = event_row["peak_date"] - pd.Timedelta(days=window_days)
    end = event_row["peak_date"] + pd.Timedelta(days=window_days)
    window = aligned.loc[aligned["Date"].between(start, end)].copy()
    if window.empty:
        return

    fig, ax = plt.subplots(figsize=(16, 7))
    for col in features:
        z = zscore_for_plot(window[col])
        linewidth = 2.5 if col == "Close" else 1.5
        ax.plot(window["Date"], z, label=f"{col} z-score", linewidth=linewidth)

    ax.axvline(event_row["peak_date"], color="crimson", linestyle="--", linewidth=2, label="Peak drift timestamp")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(
        f"Price Drift Event {int(event_row['event_id'])}: "
        f"{event_row['event_start'].date()} to {event_row['event_end'].date()} "
        f"| Peak {event_row['peak_date'].date()}"
    )
    ax.set_ylabel("Window z-score")
    ax.legend(loc="upper left", ncol=2)
    fig.tight_layout()
    out = PLOT_DIR / f"price_drift_event_{int(event_row['event_id']):03d}_cot_overlay.png"
    fig.savefig(out, dpi=180)
    plt.show()

for _, event in drift_events.head(6).iterrows():
    plot_event_window(event)

## 9. Generate timestamp-level explanation table

This table tells you where drift occurred and which COT features were most abnormal around that timestamp.

In [ ]:
cot_for_context = [c for c in plot_features if c != "Close"]
cot_stats = aligned[cot_for_context].agg(["mean", "std"]).T

context_rows = []
aligned_by_date = aligned.set_index("Date")
for _, event in drift_events.iterrows():
    date = event["peak_date"]
    if date not in aligned_by_date.index:
        continue
    values = aligned_by_date.loc[date, cot_for_context]
    z = ((values - cot_stats["mean"]) / cot_stats["std"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    top_abnormal = z.abs().dropna().sort_values(ascending=False).head(5)
    context_rows.append({
        "event_id": int(event["event_id"]),
        "event_start": event["event_start"],
        "event_end": event["event_end"],
        "peak_date": event["peak_date"],
        "peak_close": event["peak_close"],
        "peak_drift_score": event["peak_drift_score"],
        "return_5d": event["return_5d"],
        "return_20d": event["return_20d"],
        "cot_report_date_used": aligned_by_date.loc[date, "cot_report_date"],
        "cot_release_date_estimate": aligned_by_date.loc[date, "cot_release_date_estimate"],
        "cot_fill_status": aligned_by_date.loc[date, "cot_fill_status"],
        "cot_days_since_report": aligned_by_date.loc[date, "cot_days_since_report"],
        "most_abnormal_cot_context": "; ".join([f"{feature} z={score:.1f}" for feature, score in top_abnormal.items()]),
    })

event_context = pd.DataFrame(context_rows).sort_values("peak_drift_score", ascending=False)
event_context.to_csv(OUTPUT_DIR / "yahoo_price_drift_events_with_cot_context.csv", index=False)
display(event_context.head(30))

## 10. Build a return-prediction model using Yahoo + only the top 3 COT features

This section keeps the model deliberately small and interpretable:

- Yahoo Finance price-history features: returns, volatility, range, close-to-open, volume change, and trend distance
- only the top 3 COT features from the correlation table above
- target: 5-trading-day future Arabica Coffee C return

This is not the full production model. It is a compact explanation model for understanding whether the main COT features add useful signal to Yahoo price history.

In [ ]:
import json
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

top3_cot_features = cot_corr.head(3)["cot_feature"].tolist()
print("Top 3 COT features selected from drift correlations:")
display(cot_corr.head(3))

model_df = aligned.copy().sort_values("Date").reset_index(drop=True)
model_df["range_pct"] = (model_df["High"] - model_df["Low"]) / model_df["Close"]
model_df["close_to_open_pct"] = (model_df["Close"] - model_df["Open"]) / model_df["Open"]
model_df["volume_change_pct"] = model_df["Volume"].pct_change(1)
model_df["target_return_5d"] = model_df["Close"].shift(-5) / model_df["Close"] - 1.0
model_df["target_close_5d"] = model_df["Close"].shift(-5)
model_df["target_direction_5d"] = model_df["target_return_5d"] > 0

yahoo_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "realized_vol_20d",
    "close_vs_ma_63",
    "range_pct",
    "close_to_open_pct",
    "volume_change_pct",
]
model_features = yahoo_features + top3_cot_features
model_df = model_df.loc[model_df["target_return_5d"].notna()].replace([np.inf, -np.inf], np.nan).copy()

print("Model feature count:", len(model_features))
print("Model features:", model_features)
display(model_df[["Date", "Close", "target_return_5d", *model_features]].tail())

In [ ]:
def chronological_model_split(frame, train_size=0.70, valid_size=0.15):
    frame = frame.sort_values("Date").reset_index(drop=True)
    n = len(frame)
    train_end = int(n * train_size)
    valid_end = int(n * (train_size + valid_size))
    return frame.iloc[:train_end].copy(), frame.iloc[train_end:valid_end].copy(), frame.iloc[valid_end:].copy()

train_m, valid_m, test_m = chronological_model_split(model_df)
print("Rows:", {"train": len(train_m), "valid": len(valid_m), "test": len(test_m)})
print("Date ranges:")
print("train", train_m["Date"].min(), "to", train_m["Date"].max())
print("valid", valid_m["Date"].min(), "to", valid_m["Date"].max())
print("test ", test_m["Date"].min(), "to", test_m["Date"].max())

X_train = train_m[model_features]
X_valid = valid_m[model_features]
X_test = test_m[model_features]
y_train = train_m["target_return_5d"].astype(float)
y_valid = valid_m["target_return_5d"].astype(float)
y_test = test_m["target_return_5d"].astype(float)

def make_regressor_pipeline(model):
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), model_features)
        ])),
        ("model", model),
    ])

candidates = {
    "ridge": make_regressor_pipeline(Ridge(alpha=1.0)),
    "extra_trees": make_regressor_pipeline(ExtraTreesRegressor(
        n_estimators=260, max_depth=8, min_samples_leaf=15, random_state=42, n_jobs=-1
    )),
    "random_forest": make_regressor_pipeline(RandomForestRegressor(
        n_estimators=220, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1
    )),
    "hist_gradient_boosting": make_regressor_pipeline(HistGradientBoostingRegressor(
        learning_rate=0.035, max_iter=180, max_leaf_nodes=15, l2_regularization=0.05,
        early_stopping=True, random_state=42
    )),
}

validation_results = []
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_valid)
    validation_results.append({
        "model": name,
        "valid_mae": mean_absolute_error(y_valid, pred),
        "valid_rmse": root_mean_squared_error(y_valid, pred),
        "valid_r2": r2_score(y_valid, pred),
        "valid_directional_accuracy": accuracy_score((y_valid > 0).astype(int), (pred > 0).astype(int)),
    })

validation_df = pd.DataFrame(validation_results).sort_values("valid_rmse")
display(validation_df)
best_model_name = validation_df.iloc[0]["model"]
best_model = candidates[best_model_name]
print("Selected model:", best_model_name)

In [ ]:
train_valid_m = pd.concat([train_m, valid_m], ignore_index=True)
X_train_valid = train_valid_m[model_features]
y_train_valid = train_valid_m["target_return_5d"].astype(float)

best_model.fit(X_train_valid, y_train_valid)
test_pred = best_model.predict(X_test)

top3_metrics = {
    "target": "5-trading-day future Arabica Coffee C return",
    "features": model_features,
    "top3_cot_features": top3_cot_features,
    "best_model": best_model_name,
    "validation_results": validation_df.to_dict(orient="records"),
    "holdout": {
        "mae": float(mean_absolute_error(y_test, test_pred)),
        "rmse": float(root_mean_squared_error(y_test, test_pred)),
        "r2": float(r2_score(y_test, test_pred)),
        "directional_accuracy": float(accuracy_score((y_test > 0).astype(int), (test_pred > 0).astype(int))),
        "date_start": test_m["Date"].min().isoformat(),
        "date_end": test_m["Date"].max().isoformat(),
        "rows": int(len(test_m)),
    },
}

predictions = test_m[["Date", "Close", "target_close_5d", "target_return_5d", "target_direction_5d"]].copy()
predictions["predicted_return_5d"] = test_pred
predictions["predicted_close_5d"] = predictions["Close"] * (1 + predictions["predicted_return_5d"])
predictions["predicted_direction_5d"] = predictions["predicted_return_5d"] > 0
predictions["return_error"] = predictions["target_return_5d"] - predictions["predicted_return_5d"]
predictions["price_error"] = predictions["target_close_5d"] - predictions["predicted_close_5d"]

metrics_out = OUTPUT_DIR / "yahoo_cot_top3_return_model_metrics.json"
pred_out = OUTPUT_DIR / "yahoo_cot_top3_return_model_predictions.csv"
model_out = ROOT / "project" / "artifacts" / "models" / "yahoo_cot_top3_return_model.joblib"
model_out.parent.mkdir(exist_ok=True)
metrics_out.write_text(json.dumps(top3_metrics, indent=2))
predictions.to_csv(pred_out, index=False)
joblib.dump({
    "model": best_model,
    "features": model_features,
    "top3_cot_features": top3_cot_features,
    "metrics": top3_metrics,
}, model_out)

print("Holdout metrics:")
display(pd.Series(top3_metrics["holdout"]).to_frame("value"))
print("Saved:", metrics_out)
print("Saved:", pred_out)
print("Saved:", model_out)

### 5D directional accuracy matrix

This converts the 5-day return prediction into a direction call. `Up` means the 5-trading-day future return is positive; `Down/Flat` means it is zero or negative.

In [ ]:
actual_direction_5d = (predictions["target_return_5d"] > 0).astype(int)
predicted_direction_5d = (predictions["predicted_return_5d"] > 0).astype(int)
direction_labels = ["Down/Flat", "Up"]

accuracy_matrix_5d = pd.DataFrame(
    confusion_matrix(actual_direction_5d, predicted_direction_5d, labels=[0, 1]),
    index=[f"Actual {label}" for label in direction_labels],
    columns=[f"Predicted {label}" for label in direction_labels],
)
accuracy_matrix_5d_pct = accuracy_matrix_5d.div(accuracy_matrix_5d.sum(axis=1), axis=0).fillna(0)

direction_summary_5d = {
    "accuracy": float(accuracy_score(actual_direction_5d, predicted_direction_5d)),
    "precision_up": float(precision_score(actual_direction_5d, predicted_direction_5d, zero_division=0)),
    "recall_up": float(recall_score(actual_direction_5d, predicted_direction_5d, zero_division=0)),
    "f1_up": float(f1_score(actual_direction_5d, predicted_direction_5d, zero_division=0)),
    "true_down_flat": int(accuracy_matrix_5d.iloc[0, 0]),
    "false_up": int(accuracy_matrix_5d.iloc[0, 1]),
    "false_down_flat": int(accuracy_matrix_5d.iloc[1, 0]),
    "true_up": int(accuracy_matrix_5d.iloc[1, 1]),
}

top3_metrics["holdout"]["direction_accuracy_matrix_5d"] = accuracy_matrix_5d.to_dict()
top3_metrics["holdout"]["direction_accuracy_matrix_5d_pct"] = accuracy_matrix_5d_pct.round(4).to_dict()
top3_metrics["holdout"]["direction_summary_5d"] = direction_summary_5d
metrics_out.write_text(json.dumps(top3_metrics, indent=2))
joblib.dump({
    "model": best_model,
    "features": model_features,
    "top3_cot_features": top3_cot_features,
    "metrics": top3_metrics,
}, model_out)

matrix_out = OUTPUT_DIR / "yahoo_cot_top3_5d_direction_accuracy_matrix.csv"
matrix_pct_out = OUTPUT_DIR / "yahoo_cot_top3_5d_direction_accuracy_matrix_pct.csv"
summary_out = OUTPUT_DIR / "yahoo_cot_top3_5d_direction_summary.json"
accuracy_matrix_5d.to_csv(matrix_out)
accuracy_matrix_5d_pct.to_csv(matrix_pct_out)
summary_out.write_text(json.dumps(direction_summary_5d, indent=2))

print("5D direction accuracy summary:")
display(pd.Series(direction_summary_5d).to_frame("value"))
print("5D accuracy matrix - counts:")
display(accuracy_matrix_5d)
print("5D accuracy matrix - row percentages:")
display((accuracy_matrix_5d_pct * 100).round(2))

plt.figure(figsize=(6.5, 5.5))
sns.heatmap(
    accuracy_matrix_5d,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    cbar=False,
    linewidths=0.5,
    linecolor="white",
)
plt.title("Top-3 COT + Yahoo Model: 5D Direction Accuracy Matrix")
plt.xlabel("Predicted 5D direction")
plt.ylabel("Actual 5D direction")
plt.tight_layout()
plt.savefig(PLOT_DIR / "yahoo_cot_top3_5d_direction_accuracy_matrix.png", dpi=180)
plt.show()

print("Saved:", matrix_out)
print("Saved:", matrix_pct_out)
print("Saved:", summary_out)

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(predictions["Date"], predictions["target_return_5d"], label="Actual 5D return", linewidth=1.5)
plt.plot(predictions["Date"], predictions["predicted_return_5d"], label="Predicted 5D return", linewidth=1.5)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Top-3 COT + Yahoo Model: Actual vs Predicted 5D Returns")
plt.ylabel("5D return")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "yahoo_cot_top3_actual_vs_predicted_returns.png", dpi=180)
plt.show()

plt.figure(figsize=(14, 6))
plt.plot(predictions["Date"], predictions["target_close_5d"], label="Actual future close", linewidth=1.5)
plt.plot(predictions["Date"], predictions["predicted_close_5d"], label="Predicted future close", linewidth=1.5)
plt.title("Top-3 COT + Yahoo Model: Actual vs Predicted Price")
plt.ylabel("Coffee C close")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "yahoo_cot_top3_real_vs_predicted_price.png", dpi=180)
plt.show()

## 11. Add weather to the model features

This section adds Open-Meteo regional weather features to the Yahoo + top-3 COT model. It uses the available weather cache for the three important coffee regions already pulled into the project:

- Brazil Minas Gerais
- Colombia Huila
- Vietnam Dak Lak

The model does not use news features here. Weather features are rolling 7-day and 30-day values plus 30-day-vs-252-day anomalies, so each row only uses weather information available up to that date.

In [ ]:
import json
from IPython.display import Image, display

# Train the weather-enhanced model and regenerate plots. This script does not use news data.
subprocess.run([
    sys.executable,
    "project/scripts/train_top3_cot_weather_model.py",
], check=True)

weather_metrics_path = OUTPUT_DIR / "yahoo_cot_weather_return_model_metrics.json"
weather_predictions_path = OUTPUT_DIR / "yahoo_cot_weather_return_model_predictions.csv"
weather_region_impact_path = OUTPUT_DIR / "weather_region_impact.csv"
weather_model_comparison_path = OUTPUT_DIR / "yahoo_cot_weather_model_comparison.csv"
weather_drift_path = OUTPUT_DIR / "weather_regional_drift_scores.csv"

weather_metrics = json.loads(weather_metrics_path.read_text())
weather_predictions = pd.read_csv(weather_predictions_path, parse_dates=["Date"])
weather_region_impact = pd.read_csv(weather_region_impact_path)
weather_model_comparison = pd.read_csv(weather_model_comparison_path)
weather_drift = pd.read_csv(weather_drift_path, parse_dates=["Date"])

print("Weather model feature counts:")
display(pd.Series(weather_metrics["feature_counts"]).to_frame("count"))

print("Selected weather model:", weather_metrics["best_model"])
print("Selection policy:", weather_metrics.get("selection_policy", {}).get("rule", "lowest validation RMSE"))
print("Weather model holdout metrics:")
display(pd.Series(weather_metrics["holdout"]).to_frame("value"))

print("Model comparison: base models vs ensembles")
display(weather_model_comparison.sort_values("valid_rmse").head(12))

print("Regional weather impact on holdout prediction:")
display(weather_region_impact.sort_values("rmse_increase", ascending=False))

print("Saved model:", ROOT / "project" / "artifacts" / "models" / "yahoo_cot_weather_return_model.joblib")
print("Saved predictions:", weather_predictions_path)
print("Saved model comparison:", weather_model_comparison_path)
print("Saved regional weather drift scores:", weather_drift_path)

### Model and ensemble comparison

This block reads the saved comparison table from the weather-enhanced model run. It separates base models from voting ensembles, then shows which models performed best by validation RMSE, holdout RMSE, and holdout directional accuracy.

The ensemble models were tested, but the saved production-style model is only changed if the ensemble improves the validation-balanced selection rule.

### Feature-combination experiment

This experiment adds grouped average/std features for similar feature families, then evaluates reduced top-importance feature sets plus voting ensembles. The goal is to test whether engineered combinations can improve accuracy while dropping weaker raw features.

The script evaluates each feature set with the same compact model panel, then saves the best selected model and comparison outputs.


In [ ]:
# Run the feature-combination experiment. This keeps the existing weather model intact and saves a separate experiment model.
subprocess.run([
    sys.executable,
    "project/scripts/experiment_feature_combinations.py",
], check=True)

feature_combo_results_path = OUTPUT_DIR / "feature_combo_ensemble_experiment_results.csv"
feature_combo_best_path = OUTPUT_DIR / "feature_combo_ensemble_best_features.json"
feature_combo_predictions_path = OUTPUT_DIR / "feature_combo_ensemble_predictions.csv"

feature_combo_results = pd.read_csv(feature_combo_results_path)
feature_combo_best = json.loads(feature_combo_best_path.read_text())
feature_combo_predictions = pd.read_csv(feature_combo_predictions_path, parse_dates=["Date"])

print("Best feature-combo experiment:")
display(pd.Series(feature_combo_best["selected"]).to_frame("value"))

print("Selected model by feature set:")
selected_combo_cols = [
    "feature_set",
    "feature_count",
    "model",
    "model_group",
    "valid_rmse",
    "valid_directional_accuracy",
    "holdout_rmse",
    "holdout_directional_accuracy",
]
display(
    feature_combo_results.loc[feature_combo_results["selected"].eq(True), selected_combo_cols]
    .sort_values("holdout_rmse")
)

print("Best evaluated feature/model rows by holdout RMSE:")
display(
    feature_combo_results.dropna(subset=["holdout_rmse"])
    .sort_values("holdout_rmse")
    .loc[:, selected_combo_cols + ["selected"]]
    .head(15)
)

print("Saved feature-combo experiment results:", feature_combo_results_path)
print("Saved feature-combo experiment model:", ROOT / "project" / "artifacts" / "models" / "feature_combo_ensemble_model.joblib")


### Release-safe COT, crop-weather, and regime classification experiment

This section implements the release-safe COT, classification-first, crop-weather, and regime-model ideas. The next section adds walk-forward validation as a stricter out-of-time check.

- COT values are shifted to an estimated usable date after Friday release, avoiding Tuesday-report lookahead.
- The target is modeled classification-first as 5D up/down direction.
- Crop-aware weather features add heat, dry-day, heavy-rain, frost-risk, flowering/harvest, and regional stress logic.
- Regime-specific classifiers are compared against global classifiers.


In [ ]:
subprocess.run([
    sys.executable,
    "project/scripts/experiment_release_safe_regime_classification.py",
], check=True)

release_metrics_path = OUTPUT_DIR / "release_safe_regime_classification_metrics.json"
release_comparison_path = OUTPUT_DIR / "release_safe_regime_classification_comparison.csv"
release_predictions_path = OUTPUT_DIR / "release_safe_regime_classification_predictions.csv"
release_cot_audit_path = OUTPUT_DIR / "release_safe_cot_audit.csv"

release_metrics = json.loads(release_metrics_path.read_text())
release_comparison = pd.read_csv(release_comparison_path)
release_predictions = pd.read_csv(release_predictions_path, parse_dates=["Date"])
release_cot_audit = pd.read_csv(release_cot_audit_path)

print("Selected strategy:", release_metrics["selected_strategy"])
print("Best holdout classifier:")
display(pd.Series(release_metrics["selected_holdout"]).to_frame("value"))

print("Feature counts:")
display(pd.Series(release_metrics["feature_counts"]).to_frame("count"))

print("Release-safe COT audit:")
display(release_cot_audit)

print("Holdout classifier comparison:")
display(
    release_comparison.loc[release_comparison["split"].eq("holdout")]
    .sort_values("accuracy", ascending=False)
)

print("Accuracy by regime for selected classifier:")
display(
    release_predictions.groupby("regime")["correct_direction_5d"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)


In [ ]:
for plot_name in [
    "release_safe_regime_classification_accuracy.png",
    "release_safe_regime_classification_confusion_matrix.png",
    "release_safe_regime_classification_roc.png",
    "release_safe_regime_accuracy_by_regime.png",
]:
    print(plot_name)
    display(Image(filename=str(PLOT_DIR / plot_name)))


In [ ]:
for plot_name in [
    "feature_combo_ensemble_experiment_comparison.png",
    "feature_combo_ensemble_actual_vs_predicted_returns.png",
]:
    print(plot_name)
    display(Image(filename=str(PLOT_DIR / plot_name)))


In [ ]:
tested_base_models = weather_model_comparison.loc[
    weather_model_comparison["model_group"].eq("base"), "model"
].tolist()
tested_ensemble_models = weather_model_comparison.loc[
    weather_model_comparison["model_group"].eq("ensemble"), "model"
].tolist()

print("Base models tested:")
display(pd.Series(tested_base_models, name="model").to_frame())

print("Ensemble models tested:")
display(pd.Series(tested_ensemble_models, name="model").to_frame())

comparison_cols = [
    "model",
    "model_group",
    "valid_rmse",
    "valid_directional_accuracy",
    "holdout_rmse",
    "holdout_directional_accuracy",
]

print("Best by validation RMSE:")
display(weather_model_comparison[comparison_cols].sort_values("valid_rmse").head(8))

print("Best by holdout RMSE:")
display(weather_model_comparison[comparison_cols].dropna(subset=["holdout_rmse"]).sort_values("holdout_rmse").head(8))

print("Best by holdout directional accuracy:")
display(weather_model_comparison[comparison_cols].dropna(subset=["holdout_directional_accuracy"]).sort_values("holdout_directional_accuracy", ascending=False).head(8))

In [ ]:
for plot_name in [
    "yahoo_cot_weather_model_comparison.png",
    "weather_regional_drift_scores.png",
    "weather_region_impact.png",
    "yahoo_cot_weather_actual_vs_predicted_returns.png",
    "yahoo_cot_weather_real_vs_predicted_price.png",
]:
    print(plot_name)
    display(Image(filename=str(PLOT_DIR / plot_name)))

## 12. Walk-forward validation

This section reruns the model in yearly walk-forward folds. For each test year, the model trains only on older history, uses the immediately prior calendar year for model/strategy selection, then tests on the next unseen year. This is stricter than a single holdout because it shows whether performance survives changing market regimes over time.


In [ ]:
subprocess.run([
    sys.executable,
    "project/scripts/walk_forward_validation.py",
    "--start-year",
    "2012",
], check=True)

walk_metrics_path = OUTPUT_DIR / "walk_forward_release_safe_classification_metrics.json"
walk_folds_path = OUTPUT_DIR / "walk_forward_release_safe_classification_folds.csv"
walk_predictions_path = OUTPUT_DIR / "walk_forward_release_safe_classification_predictions.csv"
walk_cot_audit_path = OUTPUT_DIR / "walk_forward_release_safe_cot_audit.csv"

walk_metrics = json.loads(walk_metrics_path.read_text())
walk_folds = pd.read_csv(walk_folds_path)
walk_predictions = pd.read_csv(walk_predictions_path, parse_dates=["Date"])
walk_cot_audit = pd.read_csv(walk_cot_audit_path)

print("Walk-forward overall metrics:")
display(pd.Series(walk_metrics["overall"]).to_frame("value"))

print("Walk-forward feature counts:")
display(pd.Series(walk_metrics["feature_counts"]).to_frame("count"))

print("Release-safe COT audit used by walk-forward:")
display(walk_cot_audit)

fold_cols = [
    "test_year",
    "selected_strategy",
    "selected_model",
    "valid_global_accuracy",
    "valid_regime_accuracy",
    "test_global_accuracy",
    "test_regime_accuracy",
    "test_accuracy",
    "test_roc_auc",
    "test_rows",
]
print("Yearly walk-forward fold results:")
display(walk_folds[fold_cols])

print("Accuracy by test year from saved predictions:")
display(
    walk_predictions.groupby("test_year")["correct_direction_5d"]
    .agg(accuracy="mean", rows="count")
    .reset_index()
)


In [ ]:
for plot_name in [
    "walk_forward_release_safe_accuracy_by_year.png",
    "walk_forward_release_safe_confusion_matrix.png",
    "walk_forward_release_safe_roc.png",
    "walk_forward_release_safe_strategy_counts.png",
]:
    print(plot_name)
    display(Image(filename=str(PLOT_DIR / plot_name)))


## 13. News analysis plots with dates

These plots use `project/artifacts/outputs/news_context_by_error_date.csv` and `project/artifacts/outputs/news_articles_by_error_date.csv`. Each row is one model-error event date, with the news window around that date scored as bullish, bearish, or mixed for Arabica futures.

In [ ]:
from IPython.display import Image, display

news_points_path = OUTPUT_DIR / "news_dated_plot_points.csv"
news_points = pd.read_csv(news_points_path, parse_dates=["Date", "window_start", "window_end"])

display_cols = [
    "Date",
    "window_start",
    "window_end",
    "actual_return_5d",
    "predicted_return_5d",
    "abs_return_error",
    "article_count",
    "weighted_news_score",
    "max_weekly_strength_score",
    "news_direction",
    "news_aligned_with_actual_move",
    "strong_weekly_news_effect",
    "news_explanation",
]
display(news_points[display_cols].sort_values("abs_return_error", ascending=False).head(12))

for plot_name in [
    "news_dated_score_vs_5d_returns.png",
    "news_dated_error_alignment.png",
    "news_dated_impact_strength_bubble.png",
]:
    print(plot_name)
    display(Image(filename=str(PLOT_DIR / plot_name)))

## 14. How to read this

- Green/red vertical lines on the full plot are high-drift timestamps. Green means the 20-day return around the peak was positive; red means negative.
- `Close z-score` shows price regime on a standardized scale.
- COT lines show whether open interest, managed money, commercial hedgers, non-commercial traders, or swap dealers were also stretched or changing.
- The context CSV ranks the most abnormal COT lines at each drift timestamp.
- The top-3 return model uses Yahoo price history plus only the three strongest COT drift-correlation features.
- The weather model adds rolling regional weather features for Brazil Minas Gerais, Colombia Huila, and Vietnam Dak Lak, without using news.
- The weather model now compares multiple base models plus top-3 voting ensembles; the saved model is the best validation-balanced choice, not automatically the most complex ensemble.
- The release-safe classification experiment shifts COT availability forward, adds crop-weather logic, and compares global versus regime-specific classifiers.
- Walk-forward validation tests the release-safe classifier year by year using only past information, then reports annual accuracy, ROC AUC, confusion matrix, and strategy selection.
- The news plots mark actual event dates and compare news signal, 5D return, article count, and model error.

Generated files:

- `data/centralData/yahoo_cot_full_outer_by_date_cot_ffill.csv`
- `data/centralData/yahoo_cot_full_outer_by_date_cot_ffill.fill_report.json`
- `project/artifacts/outputs/yahoo_price_drift_events.csv`
- `project/artifacts/outputs/cot_price_drift_correlations.csv`
- `project/artifacts/outputs/yahoo_price_drift_events_with_cot_context.csv`
- `project/artifacts/outputs/yahoo_cot_top3_return_model_metrics.json`
- `project/artifacts/outputs/yahoo_cot_top3_return_model_predictions.csv`
- `project/artifacts/outputs/yahoo_cot_top3_5d_direction_accuracy_matrix.csv`
- `project/artifacts/outputs/yahoo_cot_top3_5d_direction_accuracy_matrix_pct.csv`
- `project/artifacts/outputs/yahoo_cot_top3_5d_direction_summary.json`
- `project/artifacts/outputs/yahoo_cot_weather_return_model_metrics.json`
- `project/artifacts/outputs/yahoo_cot_weather_return_model_predictions.csv`
- `project/artifacts/outputs/yahoo_cot_weather_model_comparison.csv`
- `project/artifacts/outputs/feature_combo_ensemble_experiment_results.csv`
- `project/artifacts/outputs/feature_combo_ensemble_best_features.json`
- `project/artifacts/outputs/feature_combo_ensemble_predictions.csv`
- `project/artifacts/outputs/release_safe_regime_classification_metrics.json`
- `project/artifacts/outputs/release_safe_regime_classification_comparison.csv`
- `project/artifacts/outputs/release_safe_regime_classification_predictions.csv`
- `project/artifacts/outputs/release_safe_cot_audit.csv`
- `project/artifacts/outputs/walk_forward_release_safe_classification_metrics.json`
- `project/artifacts/outputs/walk_forward_release_safe_classification_folds.csv`
- `project/artifacts/outputs/walk_forward_release_safe_classification_predictions.csv`
- `project/artifacts/outputs/walk_forward_release_safe_cot_audit.csv`
- `project/artifacts/outputs/weather_regional_drift_scores.csv`
- `project/artifacts/outputs/weather_region_impact.csv`
- `project/artifacts/outputs/news_context_by_error_date.csv`
- `project/artifacts/outputs/news_articles_by_error_date.csv`
- `project/artifacts/outputs/news_dated_plot_points.csv`
- `project/artifacts/models/yahoo_cot_top3_return_model.joblib`
- `project/artifacts/models/yahoo_cot_weather_return_model.joblib`
- `project/artifacts/models/feature_combo_ensemble_model.joblib`
- `project/artifacts/models/release_safe_regime_classification_model.joblib`
- `project/artifacts/plots/yahoo_price_drift_with_cot_lines.png`
- `project/artifacts/plots/cot_price_drift_correlations.png`
- `project/artifacts/plots/yahoo_cot_top3_5d_direction_accuracy_matrix.png`
- `project/artifacts/plots/yahoo_cot_top3_actual_vs_predicted_returns.png`
- `project/artifacts/plots/yahoo_cot_top3_real_vs_predicted_price.png`
- `project/artifacts/plots/yahoo_cot_weather_model_comparison.png`
- `project/artifacts/plots/feature_combo_ensemble_experiment_comparison.png`
- `project/artifacts/plots/feature_combo_ensemble_actual_vs_predicted_returns.png`
- `project/artifacts/plots/release_safe_regime_classification_accuracy.png`
- `project/artifacts/plots/release_safe_regime_classification_confusion_matrix.png`
- `project/artifacts/plots/release_safe_regime_classification_roc.png`
- `project/artifacts/plots/release_safe_regime_accuracy_by_regime.png`
- `project/artifacts/plots/walk_forward_release_safe_accuracy_by_year.png`
- `project/artifacts/plots/walk_forward_release_safe_confusion_matrix.png`
- `project/artifacts/plots/walk_forward_release_safe_roc.png`
- `project/artifacts/plots/walk_forward_release_safe_strategy_counts.png`
- `project/artifacts/plots/weather_regional_drift_scores.png`
- `project/artifacts/plots/weather_region_impact.png`
- `project/artifacts/plots/yahoo_cot_weather_actual_vs_predicted_returns.png`
- `project/artifacts/plots/yahoo_cot_weather_real_vs_predicted_price.png`
- `project/artifacts/plots/news_dated_score_vs_5d_returns.png`
- `project/artifacts/plots/news_dated_error_alignment.png`
- `project/artifacts/plots/news_dated_impact_strength_bubble.png`
- `project/artifacts/plots/price_drift_event_*_cot_overlay.png`